# Librerías

In [9]:
import pandas as pd
import numpy as np
import re

# Documents

Formación en las empresas:
- EAL-2024: EAL-16 a EAL-25 [18, 18a, 18b, 18c]
- EAL-2023: xxx

In [2]:
EAL_2024 = '../../Data/raw/EAL/Tablas_EAL_2024.xlsx'

In [ ]:
df_2024 = pd.read_excel(
    EAL_2024,
    sheet_name="EAL-16",
    header=None
)

He hagut de fer: pip install openpyxl (és l'engine de pandas per excel més nous)

In [5]:
df_2024.head(15)

,ENCUESTA ANUAL LABORAL,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,EAL
0,NaN,NaN,NaN,NaN,NaN,Volver al índice
1,NaN,NaN,NaN,NaN,NaN,NaN
2,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA P...,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,Año 2024. Porcentaje sobre el total de empresas.,NaN,NaN,NaN,NaN,NaN
5,NaN,TOTAL,NADA,POCO,BASTANTE,MUCHO
6,De dirección,100,10.414,18.818,40.283,30.484
7,De trabajo en equipo,100,2.224,5.822,45.125,46.829
8,De atención al público/ trato a clientes,100,5.128,13.392,37.144,44.336
9,Administrativas de oficina,100,11.117,25.313,37.788,25.782


In [14]:
def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return np.nan if x == "" else x


def to_float_es(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, float)):
        return float(x)

    x = str(x).strip()

    if x == "" or x.lower() == "nan":
        return np.nan

    # Spanish decimal format: 100,0
    if "," in x:
        x = x.replace(".", "").replace(",", ".")

    return pd.to_numeric(x, errors="coerce")


def extract_eal_tables_from_sheet(file_path, sheet_name, year=None):
    raw = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None
    )

    raw = raw.map(clean_text)

    # Detect rows where a new internal table starts
    title_pattern = re.compile(r"^EAL-\d+[A-Za-z]?\.", re.IGNORECASE)

    title_rows = []
    titles = {}

    for idx, row in raw.iterrows():
        values = row.dropna().astype(str).str.strip().tolist()

        for value in values:
            if title_pattern.search(value):
                title_rows.append(idx)
                titles[idx] = value
                break

    extracted_tables = []

    for i, start_row in enumerate(title_rows):
        end_row = title_rows[i + 1] if i + 1 < len(title_rows) else len(raw)

        block = raw.iloc[start_row:end_row].copy().reset_index(drop=True)

        title = titles[start_row]

        table_code_match = re.search(r"(EAL-\d+[A-Za-z]?)", title)
        table_code = table_code_match.group(1) if table_code_match else None

        # Try to extract year from the table text if not provided
        if year is None:
            full_text = " ".join(block.fillna("").astype(str).values.flatten())
            year_match = re.search(r"Año\s+(\d{4})", full_text)
            table_year = int(year_match.group(1)) if year_match else None
        else:
            table_year = year

        # Detect header row: TOTAL, NADA, POCO, BASTANTE, MUCHO
        expected_headers = {"TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"}

        header_row = None

        for row_idx, row in block.iterrows():
            values = row.dropna().astype(str).str.strip().str.upper().tolist()
            matches = sum(value in expected_headers for value in values)

            if matches >= 3:
                header_row = row_idx
                break

        if header_row is None:
            continue

        header_values = block.iloc[header_row]

        metric_cols = {}

        for col_idx, value in header_values.items():
            if pd.notna(value):
                value_clean = str(value).strip().upper()

                if value_clean in expected_headers:
                    metric_cols[col_idx] = value_clean

        if len(metric_cols) == 0:
            continue

        first_metric_col = min(metric_cols.keys())

        # The concept/competence column is usually before TOTAL
        possible_text_cols = list(range(first_metric_col))

        text_col = None

        for col in reversed(possible_text_cols):
            if block.iloc[header_row + 1:, col].notna().sum() > 0:
                text_col = col
                break

        if text_col is None:
            text_col = 0

        selected_cols = [text_col] + list(metric_cols.keys())

        table = block.iloc[header_row + 1:, selected_cols].copy()

        table.columns = ["competencia"] + [
            metric_cols[col].lower()
            for col in metric_cols.keys()
        ]

        table = table.dropna(how="all")
        table = table[table["competencia"].notna()]

        metric_names = [col.lower() for col in metric_cols.values()]

        for col in metric_names:
            table[col] = table[col].apply(to_float_es)

        # Keep only rows with actual numeric values
        table = table[table[metric_names].notna().any(axis=1)]

        # Add metadata
        table["year"] = table_year
        table["sheet"] = sheet_name
        table["table_code"] = table_code
        table["table_title"] = title

        # Classify company size from title
        title_upper = title.upper()

        if "5 A 49" in title_upper:
            table["company_size"] = "5 a 49 trabajadores"
        elif "50 A 499" in title_upper:
            table["company_size"] = "50 a 499 trabajadores"
        elif "MÁS DE 499" in title_upper or "MAS DE 499" in title_upper:
            table["company_size"] = "Más de 499 trabajadores"
        else:
            table["company_size"] = "Total empresas"

        extracted_tables.append(table)

    if len(extracted_tables) == 0:
        return pd.DataFrame()

    result = pd.concat(extracted_tables, ignore_index=True)

    cols_order = [
        "year",
        "sheet",
        "table_code",
        "company_size",
        "competencia",
        "total",
        "nada",
        "poco",
        "bastante",
        "mucho",
        "table_title"
    ]

    existing_cols = [col for col in cols_order if col in result.columns]

    return result[existing_cols]

In [15]:
df_eal16_2024 = extract_eal_tables_from_sheet(
    file_path=EAL_2024,
    sheet_name="EAL-16",
    year=2024
)

df_eal16_2024

,year,sheet,table_code,company_size,competencia,total,nada,poco,bastante,mucho,table_title
0,2024,EAL-16,EAL-16,Total empresas,De dirección,100,10.414,18.818,40.283,30.484,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA P...
1,2024,EAL-16,EAL-16,Total empresas,De trabajo en equipo,100,2.224,5.822,45.125,46.829,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA P...
2,2024,EAL-16,EAL-16,Total empresas,De atención al público/ trato a clientes,100,5.128,13.392,37.144,44.336,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA P...
3,2024,EAL-16,EAL-16,Total empresas,Administrativas de oficina,100,11.117,25.313,37.788,25.782,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA P...
4,2024,EAL-16,EAL-16,Total empresas,De resolución de problemas (localización de pr...,100,5.893,15.752,44.063,34.292,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA P...
...,...,...,...,...,...,...,...,...,...,...,...
65,2024,EAL-16,EAL-16f,Total empresas,En lenguas extranjeras,100,19.897,39.443,26.688,13.971,EAL-16f. EMPRESAS DEL SECTOR SERVICIOS SEGÚN G...
66,2024,EAL-16,EAL-16f,Total empresas,Básicas de cálculo y/o comunicación oral o esc...,100,23.147,35.378,29.419,12.056,EAL-16f. EMPRESAS DEL SECTOR SERVICIOS SEGÚN G...
67,2024,EAL-16,EAL-16f,Total empresas,Generales de tecnologías de la información,100,13.863,28.992,38.165,18.980,EAL-16f. EMPRESAS DEL SECTOR SERVICIOS SEGÚN G...
68,2024,EAL-16,EAL-16f,Total empresas,Profesionales de tecnologías de la información,100,27.918,36.061,24.113,11.909,EAL-16f. EMPRESAS DEL SECTOR SERVICIOS SEGÚN G...


In [16]:
df_eal16_2024['table_title'].unique()

<StringArray>
[                            'EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA  PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS',
     'EAL-16a. EMPRESAS DE 5 A 49 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA  PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS',
   'EAL-16b. EMPRESAS DE 50 A 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA  PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS',
 'EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA  PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS',
            'EAL-16d. EMPRESAS DE LA INDUSTRIA SEGÚN GRADO DE IMPORTANCIA  PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS',
         'EAL-16e. EMPRESAS DE LA CONSTRUCCIÓN SEGÚN GRADO DE IMPORTANCIA  PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPETENCIAS',
       'EAL-16f. EMPRESAS DEL SECTOR SERVICIOS SEG

Debido a que los ficheros Excel originales contienen varias tablas dentro de una misma hoja, con títulos, notas y celdas vacías, se realizó una selección manual de las tablas relevantes para el análisis. Posteriormente, la información seleccionada se estructuró en formato tabular limpio, manteniendo variables de trazabilidad como el año, la hoja y el código de tabla de origen.